[![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/Gad-gg-23/Identificador_Rostros_Modelo/blob/main/kaggle/Entrenamiento_Modelo_Kaggle.ipynb)

# Entrenamiento del identificador de rostros (Kaggle)

Version para Kaggle del notebook de `colab/`. Antes de correr, en el panel derecho (**Settings**):

1. **Accelerator**: `GPU T4 x2` o `GPU P100`.
2. **Internet**: `On` (hace falta para clonar el repo y bajar el dataset de Hugging Face).
3. **Add-ons > Secrets**: agrega un secreto llamado `HF_TOKEN` con un token de lectura de una cuenta
   miembro de la organizacion `escom-identificador-rostros`, y marca la casilla para adjuntarlo a este notebook.

**Donde queda el progreso:** todo lo que se escribe en `/kaggle/working/` (rostros preprocesados y checkpoints)
se guarda como *Output* del notebook cuando haces **Save Version**. Lo mejor es usar
**Save Version > Save & Run All (Commit)**: corre en segundo plano hasta 12 h aunque cierres el navegador.

In [ ]:
# Verifica GPU e internet
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, socket
print("CUDA disponible:", torch.cuda.is_available())
try:
    socket.create_connection(("github.com", 443), timeout=5)
    print("Internet: OK")
except OSError:
    print("Internet: NO -> activalo en Settings > Internet")

## 0. Clonar el repo e instalar dependencias

Kaggle ya trae torch, torchvision, pandas, scikit-learn, matplotlib, etc. Solo falta `facenet-pytorch`,
que se instala con `--no-deps` para que no baje una version vieja de torch y rompa la de Kaggle.

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/Identificador_Rostros_Modelo
!git clone -q https://github.com/Gad-gg-23/Identificador_Rostros_Modelo.git
%cd /kaggle/working/Identificador_Rostros_Modelo
!pip install -q --no-deps facenet-pytorch
!pip install -q -U huggingface_hub pyyaml

## 1. Login a Hugging Face

Lee el token del secreto `HF_TOKEN` (Add-ons > Secrets). Si no lo configuraste, cae al login interactivo,
que funciona en sesion interactiva pero **no** en *Save & Run All*.

In [ ]:
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = token
    login(token=token)
    print("Login con el secreto HF_TOKEN: OK")
except Exception as e:
    print("No se pudo leer el secreto HF_TOKEN:", e)
    login()

## 2. Configurar rutas para Kaggle

- Fotos originales -> `/tmp/data` (temporal, no se guarda en el Output para no inflarlo).
- Rostros preprocesados y checkpoints -> `/kaggle/working/` (se guardan en el Output).

Se escribe un `config_kaggle.yaml` aparte para no tocar el `config.yaml` del repo.

In [ ]:
import yaml
from pathlib import Path

DATA_DIR = Path('/tmp/data')
SALIDA = Path('/kaggle/working')
ROSTROS_DIR = SALIDA / 'rostros'
CKPT_DIR = SALIDA / 'checkpoints'
for p in (DATA_DIR, ROSTROS_DIR, CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)

with open('config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['datos']['raiz'] = str(DATA_DIR)
cfg['datos']['rostros_raiz'] = str(ROSTROS_DIR)
cfg['salida']['checkpoints'] = str(CKPT_DIR)
cfg['salida']['mejor_modelo'] = str(CKPT_DIR / 'mejor_modelo.pt')
cfg['salida']['log_csv'] = str(CKPT_DIR / 'historial_entrenamiento.csv')

# Si la GPU se queda sin memoria (CUDA out of memory), baja el batch, p. ej.:
# cfg['entrenamiento']['batch_size'] = 32

CONFIG = 'config_kaggle.yaml'
with open(CONFIG, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)

print(yaml.safe_dump(cfg, allow_unicode=True))

## 3. (Opcional) Reanudar desde una version anterior

Si una corrida anterior se corto, ve a **Add Input > Your Work / Notebook Output** y agrega el Output de esa
version. Esta celda busca ahi `rostros/` y `checkpoints/` y los copia a `/kaggle/working/`, asi el
preprocesamiento se salta las fotos ya hechas y `entrenar.py --resume` sigue donde se quedo.
Si no agregaste nada, no hace nada.

In [ ]:
import shutil

encontrado = False
for estado in Path('/kaggle/input').rglob('checkpoints/ultimo_estado.pt'):
    origen = estado.parent.parent
    print('Reanudando desde:', origen)
    shutil.copytree(origen / 'checkpoints', CKPT_DIR, dirs_exist_ok=True)
    if (origen / 'rostros').exists():
        shutil.copytree(origen / 'rostros', ROSTROS_DIR, dirs_exist_ok=True)
    encontrado = True
    break

if not encontrado:
    print('No hay checkpoints previos en /kaggle/input: se entrena desde cero.')

## 4. Descargar el dataset

In [ ]:
!python descargar_datos.py --destino /tmp/data
!du -sh /tmp/data

## 5. Detectar y alinear rostros (MTCNN)

In [ ]:
!python src/preprocesar_rostros.py --config config_kaggle.yaml

## 6. Entrenar (2 fases: cabeza ArcFace, luego fine-tuning)

In [ ]:
!python src/entrenar.py --config config_kaggle.yaml --resume

## 7. Evaluar

In [ ]:
!python src/evaluar.py --config config_kaggle.yaml

from IPython.display import Image, display
matriz = CKPT_DIR / 'matriz_confusion.png'
if matriz.exists():
    display(Image(filename=str(matriz)))

## 8. Descargar el checkpoint entrenado

Despues de **Save Version**, el modelo aparece en la pestana **Output** del notebook
(`checkpoints/mejor_modelo.pt`). En sesion interactiva tambien puedes bajarlo con el enlace de abajo.

In [ ]:
from IPython.display import FileLink
%cd /kaggle/working
display(FileLink('checkpoints/mejor_modelo.pt'))
%cd /kaggle/working/Identificador_Rostros_Modelo

## 9. Probar una foto nueva

Kaggle no tiene el boton de subir archivo de Colab. Sube tu foto como dataset
(**Add Input > Upload**) y pon su ruta en `FOTO`. Si la dejas vacia, se prueba con una foto
del dataset.

In [ ]:
FOTO = ''  # p. ej. '/kaggle/input/mis-fotos/prueba.jpg'

if not FOTO:
    fotos = sorted(p for p in DATA_DIR.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    FOTO = str(fotos[-1])
print('Probando con:', FOTO)
!python src/inferir.py --imagen "{FOTO}" --config config_kaggle.yaml

## 10. Limpieza antes de guardar

El clon del repo no hace falta en el Output. Se borra para que solo queden `rostros/` y `checkpoints/`.

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/Identificador_Rostros_Modelo
!du -sh /kaggle/working/*